# Reusable Template: Simple Linear Regression (from-scratch GD + sklearn)

Drop in any two continuous columns (or a CSV with two numeric columns) and obtain:
1. From-scratch gradient-descent fit (centered for stability)
2. scikit-learn verification
3. Closed-form solution
4. Residual diagnostics
5. Parameterised Monte-Carlo sensitivity simulation

**Audience adaptation:** change the narrative cells at the bottom according to whether the reader is technical, domain-expert, executive, or non-specialist.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Configuration — edit these values

In [ ]:
# Path to a CSV that contains the two columns of interest
DATA_PATH   = 'data/heights.csv'
X_COL       = 'height'          # predictor column name
Y_COL       = 'weight'          # response column name
X_LABEL     = 'Height (inches)'
Y_LABEL     = 'Weight (lb)'
TITLE       = 'Height → Weight Linear Regression'

# Gradient-descent hyper-parameters (centered data)
LEARNING_RATE = 0.001
N_ITER        = 2000

# Simulation defaults
NOISE_STD   = 0.0
SAMPLE_FRAC = 1.0
N_REPS      = 25

## Core functions (copy-paste ready)

In [ ]:
def get_gradient_at_b(x, y, m, b):
    return -2.0 * np.mean(y - (m * x + b))

def get_gradient_at_m(x, y, m, b):
    return -2.0 * np.mean(x * (y - (m * x + b)))

def step_gradient(x, y, b, m, lr):
    b_new = b - lr * get_gradient_at_b(x, y, m, b)
    m_new = m - lr * get_gradient_at_m(x, y, m, b)
    return b_new, m_new

def gradient_descent(x, y, lr=0.001, n_iter=2000):
    """Assumes x, y are already centered (or small scale). Returns (b, m)."""
    b, m = 0.0, 0.0
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    for _ in range(n_iter):
        b, m = step_gradient(x, y, b, m, lr)
    return b, m

def closed_form(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.cov(x, y, ddof=0)[0, 1] / np.var(x)
    b = y.mean() - m * x.mean()
    return b, m

def fit_all(x, y, lr=0.001, n_iter=2000):
    """Return dict with GD (centered), sklearn, and closed-form results."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    # centered GD
    b_c, m_c = gradient_descent(x - x.mean(), y - y.mean(), lr, n_iter)
    b_gd = y.mean() - m_c * x.mean()
    m_gd = m_c
    # sklearn
    model = LinearRegression().fit(x.reshape(-1, 1), y)
    # closed form
    b_cf, m_cf = closed_form(x, y)
    return {
        'gd': (b_gd, m_gd),
        'sklearn': (model.intercept_, model.coef_[0]),
        'closed': (b_cf, m_cf),
        'r2': model.score(x.reshape(-1, 1), y),
        'model': model
    }

## Load data & fit

In [ ]:
df = pd.read_csv(DATA_PATH)
X = df[X_COL].values
y = df[Y_COL].values

results = fit_all(X, y, LEARNING_RATE, N_ITER)
print('Centered GD : m={:.4f}, b={:.4f}'.format(*results['gd'][::-1]))
print('sklearn     : m={:.4f}, b={:.4f}'.format(results['sklearn'][1], results['sklearn'][0]))
print('Closed-form : m={:.4f}, b={:.4f}'.format(results['closed'][1], results['closed'][0]))
print('R²          : {:.4f}'.format(results['r2']))

## Visualisation & residuals

In [ ]:
m, b = results['sklearn'][1], results['sklearn'][0]
yhat = m * X + b
resid = y - yhat

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X, y, alpha=0.6, edgecolor='k', linewidth=0.3)
xx = np.linspace(X.min(), X.max(), 100)
axes[0].plot(xx, m*xx + b, 'r-', lw=2)
axes[0].set_xlabel(X_LABEL); axes[0].set_ylabel(Y_LABEL)
axes[0].set_title(TITLE + f'  (R²={results["r2"]:.3f})')

axes[1].scatter(yhat, resid, alpha=0.6, edgecolor='k', linewidth=0.3)
axes[1].axhline(0, color='r', ls='--')
axes[1].set_xlabel('Fitted'); axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs Fitted')
plt.tight_layout(); plt.show()

## Monte-Carlo sensitivity simulation

In [ ]:
def one_rep(X, y, lr, n_iter, noise, frac):
    n = len(X)
    idx = np.random.choice(n, size=max(15, int(n*frac)), replace=False)
    Xs, ys = X[idx], y[idx] + (np.random.normal(0, noise, size=len(idx)) if noise else 0)
    res = fit_all(Xs, ys, lr, n_iter)
    return res['sklearn'][1], res['sklearn'][0], res['r2']

ms, bs, r2s = zip(*[one_rep(X, y, LEARNING_RATE, N_ITER, NOISE_STD, SAMPLE_FRAC)
                    for _ in range(N_REPS)])
print(f'Slope   {np.mean(ms):.4f} ± {np.std(ms):.4f}')
print(f'Intercept {np.mean(bs):.2f} ± {np.std(bs):.2f}')
print(f'R²      {np.mean(r2s):.4f} ± {np.std(r2s):.4f}')

## Audience-ready summary snippets (edit as needed)

**For a technical supervisor**  
“Simple linear regression of weight on height yields slope ≈ 3.43 lb/in, intercept ≈ −106, R² ≈ 0.31. From-scratch centered gradient descent, the normal equations and sklearn agree to four decimal places. Residuals show no strong non-linear pattern.”

**For an executive / decision maker**  
“Taller players tend to be heavier: each additional inch of height is associated with about 3.4 pounds more weight. Height alone explains roughly 31 % of the differences in weight we see in this sample.”

**For a non-specialist audience**  
“We drew a straight line through a cloud of points that show players’ heights and weights. The line slopes upward, telling us that taller players are generally heavier. The line is a useful rough guide, but many other factors also affect weight.”
